# PKM_RF Triage Environment Setup & Fast Binary Package Installation (`setup.ipynb`)

Welcome to the **PKM_RF Emergency Severity Index (ESI) Triage Machine Learning System**.
This notebook initializes your environment, automatically installs pre-compiled Linux binary Python and R packages (including `caret`, `lightgbm`, `optuna`, `pROC`, `e1071`), validates project configuration, checks dataset availability, creates output directories, and provides execution triggers for all pipeline notebooks.

--- 
### Project Architecture & Notebook Inventory

1. **Configuration**: [`config/triage_conf.json`](file:///home/apt2736/PKM_RF/config/triage_conf.json)
   - Central project configuration defining dataset paths, target column (`esi`), train/val/test split ratios, random seed, and resampling ratios.

2. **Engineered Feature Distribution Plots**: [`models/plot_engineered.ipynb`](file:///home/apt2736/PKM_RF/models/plot_engineered.ipynb)
   - Generates distribution graphs for 26 feature-engineered inputs across ESI classes `1` through `5` in `plots/engineered_feature_distributions/`.

3. **3-Tier Hierarchical Training**: [`models/train_hierarchical_lightgbm_layers.ipynb`](file:///home/apt2736/PKM_RF/models/train_hierarchical_lightgbm_layers.ipynb)
   - Trains 4 LightGBM sub-models with layer-exclusive features (`deploy/*.rds`).

4. **Combined Master Pipeline Benchmark**: [`models/combined_hierarchical_triage_pipeline.ipynb`](file:///home/apt2736/PKM_RF/models/combined_hierarchical_triage_pipeline.ipynb)
   - Evaluates Soft Probabilistic Joint Product predictions vs Hard Case-Selector routing on the 1% holdout test set.

5. **Native C Transpilation**: [`models/transpile_soft_pipeline_to_c.ipynb`](file:///home/apt2736/PKM_RF/models/transpile_soft_pipeline_to_c.ipynb)
   - Transpiles all sub-models into zero-dependency C code (`deploy/triage_soft_pipeline.c`).

6. **Layer 1 Anomaly Detection Benchmark**: [`models/benchmark_layer1_anomaly_detectors.ipynb`](file:///home/apt2736/PKM_RF/models/benchmark_layer1_anomaly_detectors.ipynb)
   - Benchmarks One-Class SVM, Isolation Forest, K-Means, and LightGBM baseline.

7. **Automated Resampling Calibration**: [`models/calibrate_resampling_ratios_5fold_cv.ipynb`](file:///home/apt2736/PKM_RF/models/calibrate_resampling_ratios_5fold_cv.ipynb)
   - Optimizes Layer 1 class weight multipliers and Layer 2 resampling ratios using 5-Fold Stratified CV with Optuna.

In [ ]:
# ---------------------------------------------------------
# Step 1: Python Dependencies Check & Auto-Installation
# ---------------------------------------------------------
import sys
import subprocess
import os
import json
print("=== Step 1: Python Environment Info ===")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Working Dir:    {os.getcwd()}")
required_py_pkgs = ['numpy', 'pandas', 'scipy', 'sklearn', 'lightgbm', 'optuna', 'matplotlib', 'seaborn', 'rpy2']
missing_py = []
for pkg in required_py_pkgs:
    try:
        __import__(pkg)
        print(f"  [OK] Python package: {pkg}")
    except ImportError:
        missing_py.append(pkg)
        print(f"  [MISSING] Python package: {pkg}")
if missing_py:
    print(f"\nInstalling missing Python packages: {missing_py} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_py)
    print("Python packages installed successfully!")
# Ensure output directories exist
for folder in ['deploy', 'reports', 'plots', 'plots/engineered_feature_distributions']:
    os.makedirs(folder, exist_ok=True)
    print(f"  [DIR OK] {folder}/")

In [ ]:
# ---------------------------------------------------------
# Step 2: Validate Project Configuration & Data Source
# ---------------------------------------------------------
config_file = "config/triage_conf.json"
if os.path.exists(config_file):
    with open(config_file, "r") as f:
        config = json.load(f)
    print("\n=== Step 2: Project Configuration Validated ===")
    print(f"  Data Source: {config['path']['data_source']}")
    print(f"  Target Col:  {config['classes']['target_col']}")
    
    data_path = config['path']['data_source']
    if os.path.exists(data_path):
        print(f"  [DATASET OK] Found data file: {data_path}")
    else:
        print(f"  [WARNING] Data file not found at: {data_path}")
else:
    print(f"[ERROR] Configuration file missing: {config_file}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Initialize rpy2 for R Execution
# ---------------------------------------------------------
try:
    %load_ext rpy2.ipython
    print("rpy2 extension loaded successfully.")
except Exception as e:
    print("Note on rpy2 initialization:", e)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Fast Pre-compiled Binary R Package Installer (Posit PPM Linux Mirror)
# ---------------------------------------------------------
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
r_pkgs <- c("jsonlite", "dplyr", "ggplot2", "tidyr", "pROC", "lightgbm", "e1071", "caret")
cat("=== Step 4: Auditing & Installing Pre-compiled R Packages ===\n")
missing_r <- c()
for (pkg in r_pkgs) {
  if (suppressWarnings(require(pkg, character.only = TRUE, quietly = TRUE))) {
    cat(sprintf("  [OK] R package: %s\n", pkg))
  } else {
    cat(sprintf("  [MISSING] R package: %s\n", pkg))
    missing_r <- c(missing_r, pkg)
  }
}
if (length(missing_r) > 0) {
  cat(sprintf("\nInstalling pre-compiled R binary packages: %s ...\n", paste(missing_r, collapse=", ")))
  install.packages(missing_r, dependencies = TRUE)
  cat("All R packages installed successfully!\n")
}

In [ ]:
# ---------------------------------------------------------
# Step 5: Pipeline Execution Triggers
# ---------------------------------------------------------
print("SETUP COMPLETED SUCCESSFULLY!")
print("All Python & R dependencies (including caret) are installed and ready.")